In [1]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")   # a harmless warning from LAL
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch
from scipy import stats
from pycbc.waveform import get_fd_waveform
from pycbc.detector.ground import Detector

# Challenge 2: False Alarm and Detection Probability (final version)

Two possible binary mergers, **A** and **B**, may be in the Hanford (H1) data. We know the time **and the phase** of each signal, so the SNR compares the data directly with the template:
$$\rho = \frac{(d|h)}{\sqrt{(h|h)}} .$$
There is no absolute value, so **$\rho$ can be negative**.

We measured $\rho_{\rm obs}$ for each signal, and we ask:
- **False alarm probability (FAP):** If the data is only noise, how likely is it to get $\rho \geq \rho_{\rm obs}$?
- **Detection probability (DP):** If the data contains the signal, how likely is it to get $\rho \geq \rho_{\rm obs}$?

We claim a detection if $\mathrm{FAP} < \mathrm{FAP}_{\rm th} = 0.01$.

**Task:**
1. Estimate the PSD from the off-source data with Welch's method.
2. Compute the FAP and the DP of both signals by simulating noise.
3. Estimate their errors, and decide which signal is a detection.

## What you are given

**You are given:** the parameters of the binaries, $\rho_{\rm obs}$, and 64 s of off-source data (data without any signal) in the file `data/challenge2_final_offsource.npz`.

In [2]:
# parameters of the two binaries, and the SNR we measured for each of them
signals = {
    "A": dict(m_1=30.5, m_2=25.3, distance=2500, ra=0.83, dec=0.77, rho_obs=3.0),
    "B": dict(m_1=14.0, m_2=11.0, distance=2000, ra=2.20, dec=-0.50, rho_obs=2.0),
}

gps_time = 1186741861   # time of the signals

# detection threshold
fap_th = 0.01

# off-source data: 64 s of H1 data, 10 minutes before the signals
fs = 4096               # Hz
gps_start = gps_time - 600
# It was fetched from GWOSC with:
# from gwpy.timeseries import TimeSeries
# offsource = TimeSeries.fetch_open_data("H1", gps_start, gps_start + 64, sample_rate=fs, cache=True).value
offsource = np.load("data/challenge2_final_offsource.npz")["offsource"]

print("offsource has shape", offsource.shape)

offsource has shape (262144,)


## Step 1: Estimate the PSD

Call `welch` on the off-source data with pieces of $T = 4$ s. The PSD is then given at the frequencies `freqs`, which are spaced by `df` $= 1/T$. Stick to these settings so the frequency grid matches the simulated signal.

In [ ]:
T = 4   # s

# TO DO: estimate the PSD from the off-source data with welch
freqs, psd = ...
df = freqs[1] - freqs[0]

plt.loglog(freqs, psd)
plt.xlim(10, fs / 2)
plt.xlabel("frequency (Hz)")
plt.ylabel(r"$S_n(f)$ (1/Hz)")
plt.show()

## Step 2: The signals and the SNR

The signals are given: we project each waveform onto the H1 detector. Write the inner product and the SNR, using only frequencies between 20 Hz and 500 Hz:
$$(a|b) = 4\, \mathrm{Re} \sum_{f} \frac{a(f)\, b^{*}(f)}{S_n(f)}\, \Delta f .$$

In [3]:
signals.items()

dict_items([('A', {'m_1': 30.5, 'm_2': 25.3, 'distance': 2500, 'ra': 0.83, 'dec': 0.77, 'rho_obs': 3.0}), ('B', {'m_1': 14.0, 'm_2': 11.0, 'distance': 2000, 'ra': 2.2, 'dec': -0.5, 'rho_obs': 2.0})])

In [ ]:
# TO DO: select the frequencies between 20 Hz and 500 Hz
band = ...

def inner_prod(a, b):
    # TO DO
    return ...

def snr(d, h):
    # TO DO
    return ...

def get_signal(s):
    Fp, Fc = Detector("H1").antenna_pattern(s["ra"], s["dec"], 0, gps_time)
    hp, hc = get_fd_waveform(approximant="IMRPhenomD", mass1=s["m_1"], mass2=s["m_2"], distance=s["distance"],
                             delta_f=df, f_lower=20, f_final=fs / 2)
    return (hp.numpy() * Fp + hc.numpy() * Fc)[:len(freqs)]

# add the waveform and the optimal snr to the dictionary
for name, s in signals.items():
    s["h"] = get_signal(s)
    # TO DO
    s["rho_opt"] = ...
    print(f"signal {name}: rho_opt = {s['rho_opt']:.2f}, rho_obs = {s['rho_obs']}")

## Step 3: Simulate many experiments

One realisation of the noise in frequency domain is
$$n(f) = \sqrt{\frac{T\, S_n(f)}{4}}\,\big(x + i\, y\big), \qquad x,\, y \sim \mathcal{N}(0, 1) .$$

For each signal, repeat the experiment `N_sim` times and compute the SNR with only noise, $d = n$, and with the signal, $d = n + h$.

In [ ]:
def simulate_noise(rng):
    # TO DO
    return ...

N_sim = 10000
rng = np.random.default_rng(1234)

# add the SNR arrays to the dictionary
for s in signals.values():
    s["rho_noise"] = np.zeros(N_sim)
    s["rho_signal"] = np.zeros(N_sim)
    for k in range(N_sim):
        # TO DO
        n = ...
        s["rho_noise"][k] = ...
        s["rho_signal"][k] = ...

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
for ax, (name, s) in zip(axes, signals.items()):
    ax.hist(s["rho_noise"], bins=50, density=True, alpha=0.6, label="only noise")
    ax.hist(s["rho_signal"], bins=50, density=True, alpha=0.6, label="noise + signal")
    ax.axvline(s["rho_obs"], color="crimson", ls="--", label=r"$\rho_{\rm obs}$")
    ax.set_title(f"signal {name}")
    ax.set_xlabel(r"$\rho$")
axes[0].set_ylabel("pdf")
axes[0].legend(loc="upper left", fontsize=8)
plt.show()

## Step 4: FAP and DP with their errors

Both the FAP and DP are the fraction of simulations above $\rho_{\rm obs}$. As in Lecture 7, the error of a fraction $p$ measured with $N$ samples is
$$\sigma_p = \sqrt{\frac{p(1-p)}{N}} .$$

In [ ]:
def error(p, N):
    # TO DO
    return ...

for name, s in signals.items():
    # TO DO
    fap = ...
    dp = ...
    decision = ...
    print(f"signal {name}: FAP = {fap:.5f} +- {error(fap, N_sim):.5f}, "
          f"DP = {dp:.4f} +- {error(dp, N_sim):.4f}  -->  {decision}")